# Preprocessing Data Pemain Sepak Bola

Notebook ini menjalankan preprocessing dari data Transfermarkt dan FBref yang sudah tersedia. Notebook tidak menjalankan scraping ulang.

## 1. Import Library dan Konfigurasi

In [1]:
%pip install rapidfuzz unidecode

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from pathlib import Path
import importlib
import json
import math
import re
import subprocess
import sys
import warnings

import numpy as np
import pandas as pd


def ensure_package(import_name, package_name=None):
    package_name = package_name or import_name
    try:
        return importlib.import_module(import_name)
    except ModuleNotFoundError:
        print(f'Menginstall dependency: {package_name}')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', package_name])
        return importlib.import_module(import_name)


rapidfuzz = ensure_package('rapidfuzz')
unidecode_module = ensure_package('unidecode')
from rapidfuzz import fuzz, process
from unidecode import unidecode

warnings.filterwarnings('ignore')

PROJECT_ROOT = Path.cwd()

RAW_PLAYERS_FILE = PROJECT_ROOT / 'data' / 'raw' / 'players_raw.csv'
FBREF_FILE = PROJECT_ROOT / 'data' / 'interim' / 'fbref_player_stats.csv'

PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
MODEL_DIR = PROJECT_ROOT / 'data' / 'model'
INTERIM_DIR = PROJECT_ROOT / 'data' / 'interim'

CLEAN_FILE = PROCESSED_DIR / 'transfermarkt_clean.csv'
PREPROCESSING_REPORT_FILE = PROCESSED_DIR / 'preprocessing_report.csv'
MODEL_FILE = MODEL_DIR / 'players_model.csv'
FEATURE_LIST_FILE = MODEL_DIR / 'feature_list.json'
DATASET_SUMMARY_FILE = MODEL_DIR / 'dataset_summary.csv'
MATCHING_FILE = INTERIM_DIR / 'player_matching_result.csv'
UNMATCHED_FILE = INTERIM_DIR / 'unmatched_players.csv'

for folder in [PROCESSED_DIR, MODEL_DIR, INTERIM_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
MIN_SEASON = 2017
MAX_SEASON = 2024
TARGET_COLUMN = 'market_value_category'
TARGET_LABELS = ['Rendah', 'Menengah', 'Tinggi']

REQUIRED_TRANSFERMARKT_COLUMNS = [
    'player_id', 'player_name', 'player_url', 'pos_category', 'position_detail',
    'age', 'nationality', 'height_m', 'preferred_foot', 'club',
    'club_total_mv_raw', 'league', 'league_rank', 'season',
    'market_value_str', 'market_value_mio'
]

FORBIDDEN_FEATURES = [
    'market_value_mio', 'market_value_str', 'market_value_category',
    'value_category', 'label', 'target', 'player_id', 'player_name',
    'player_url', 'position_detail', 'mv_growth_rate'
]

FBREF_FEATURES = [
    'matches_played', 'starts', 'minutes', 'goals', 'assists',
    'non_penalty_goals', 'yellow_cards', 'red_cards', 'shots_total',
    'shots_on_target', 'fouls_committed', 'fouls_drawn', 'saves',
    'clean_sheets', 'goals_against', 'shots_on_target_against',
    'goals_per_90', 'assists_per_90', 'goal_assist_per_90',
    'shots_per_90', 'shots_on_target_per_90', 'cards_per_90',
    'starts_rate', 'save_pct', 'clean_sheet_pct', 'has_performance_stats',
    'goal_contribution_per_90', 'attacking_output_index',
    'discipline_index', 'goalkeeper_output_index'
]

print('Project root:', PROJECT_ROOT)

Project root: c:\KULIAH\SEMESTER 6\BIG DATA\UAS\Tugas_UAS_Big-Data


## 2. Helper Validasi dan Parsing

In [3]:
def require_non_empty_file(path):
    if not path.exists():
        raise FileNotFoundError(f'File tidak ditemukan: {path}')
    if path.stat().st_size == 0:
        raise ValueError(f'File kosong: {path}')
    return path


def validate_columns(df, required_columns, dataset_name):
    missing = [col for col in required_columns if col not in df.columns]
    if missing:
        raise ValueError(f'Kolom wajib tidak tersedia pada {dataset_name}: {missing}')


def parse_market_value_to_mio(value):
    if pd.isna(value):
        return np.nan
    if isinstance(value, (int, float, np.integer, np.floating)):
        return float(value)
    text = str(value).strip().lower()
    if text in ['', '-', 'nan', 'none']:
        return np.nan
    text = text.replace('€', '').replace(',', '').replace(' ', '')
    match = re.search(r'([0-9]+(?:\.[0-9]+)?)', text)
    if not match:
        return np.nan
    number = float(match.group(1))
    if 'bn' in text or 'b' in text:
        return number * 1000.0
    if 'm' in text:
        return number
    if 'k' in text or 'th' in text:
        return number / 1000.0
    return number


def normalize_text(value):
    if pd.isna(value):
        return ''
    text = unidecode(str(value).lower())
    text = re.sub(r'[^a-z0-9]+', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()


def safe_divide(numerator, denominator):
    numerator = pd.to_numeric(numerator, errors='coerce')
    denominator = pd.to_numeric(denominator, errors='coerce')
    result = numerator / denominator.replace(0, np.nan)
    return result.replace([np.inf, -np.inf], np.nan).fillna(0)


def assign_market_value_category(value):
    if 5 <= value < 15:
        return 'Rendah'
    if 15 <= value <= 35:
        return 'Menengah'
    if value > 35:
        return 'Tinggi'
    return np.nan


def assign_previous_value_category(value):
    if pd.isna(value) or value <= 0:
        return 'No History'
    if value < 5:
        return 'Below 5'
    return assign_market_value_category(value)

## 3. Load dan Validasi Input

In [4]:
require_non_empty_file(RAW_PLAYERS_FILE)
players_raw = pd.read_csv(RAW_PLAYERS_FILE)
validate_columns(players_raw, REQUIRED_TRANSFERMARKT_COLUMNS, 'players_raw.csv')

fbref_available = FBREF_FILE.exists() and FBREF_FILE.stat().st_size > 0
if fbref_available:
    fbref_raw = pd.read_csv(FBREF_FILE)
else:
    fbref_raw = pd.DataFrame()

print('Rows Transfermarkt raw:', len(players_raw))
print('Rows FBref raw:', len(fbref_raw))
print('FBref tersedia:', fbref_available)

Rows Transfermarkt raw: 30024
Rows FBref raw: 22425
FBref tersedia: True


## 4. Cleaning Transfermarkt

In [5]:
def clean_transfermarkt(df):
    report = []
    clean = df.copy()
    report.append({'metric': 'raw_rows', 'value': len(clean)})

    clean['club_total_mv_mio'] = clean['club_total_mv_raw'].apply(parse_market_value_to_mio)

    numeric_columns = ['season', 'age', 'height_m', 'league_rank', 'market_value_mio', 'club_total_mv_mio']
    for col in numeric_columns:
        clean[col] = pd.to_numeric(clean[col], errors='coerce')

    clean = clean[clean['season'].between(MIN_SEASON, MAX_SEASON)].copy()
    report.append({'metric': 'rows_after_season_filter', 'value': len(clean)})

    before_drop_market_value = len(clean)
    clean = clean.dropna(subset=['market_value_mio']).copy()
    report.append({'metric': 'rows_dropped_missing_market_value', 'value': before_drop_market_value - len(clean)})

    before_duplicates = len(clean)
    clean = clean.sort_values(['player_id', 'season']).drop_duplicates(['player_id', 'season'], keep='last')
    report.append({'metric': 'duplicate_player_season_rows_dropped', 'value': before_duplicates - len(clean)})

    categorical_columns = [
        'player_name', 'player_url', 'shirt_number', 'pos_category', 'position_detail',
        'date_of_birth', 'nationality', 'preferred_foot', 'club', 'league', 'market_value_str'
    ]
    for col in categorical_columns:
        if col in clean.columns:
            clean[col] = clean[col].astype('object').where(clean[col].notna(), 'Unknown')
            clean[col] = clean[col].replace({'': 'Unknown', 'nan': 'Unknown', 'None': 'Unknown'})

    numeric_impute_columns = ['age', 'height_m', 'league_rank', 'club_total_mv_mio']
    for col in numeric_impute_columns:
        position_median = clean.groupby('pos_category')[col].transform('median')
        overall_median = clean[col].median()
        if pd.isna(overall_median):
            overall_median = 0
        clean[col] = clean[col].fillna(position_median).fillna(overall_median)

    history = clean[['player_id', 'season', 'market_value_mio']].copy()
    prev = history.rename(columns={'market_value_mio': 'prev_season_mv'}).copy()
    prev['season'] = prev['season'] + 1
    two_prev = history.rename(columns={'market_value_mio': 'two_seasons_ago_mv'}).copy()
    two_prev['season'] = two_prev['season'] + 2

    clean = clean.merge(prev, on=['player_id', 'season'], how='left')
    clean = clean.merge(two_prev, on=['player_id', 'season'], how='left')
    clean = clean.sort_values(['player_id', 'season']).copy()
    clean['mv_history_count'] = clean.groupby('player_id').cumcount()

    before_min_value = len(clean)
    clean = clean[clean['market_value_mio'] >= 5].copy()
    report.append({'metric': 'rows_dropped_below_5_mio', 'value': before_min_value - len(clean)})

    clean[TARGET_COLUMN] = clean['market_value_mio'].apply(assign_market_value_category)
    if clean[TARGET_COLUMN].isna().any():
        raise ValueError('Target market_value_category gagal dibuat untuk sebagian row.')

    report.append({'metric': 'clean_rows', 'value': len(clean)})
    return clean.reset_index(drop=True), report


transfermarkt_clean, preprocessing_report = clean_transfermarkt(players_raw)
transfermarkt_clean.head()

,player_id,player_name,player_url,shirt_number,pos_category,position_detail,age,date_of_birth,nationality,height_m,...,league,league_rank,season,market_value_str,market_value_mio,club_total_mv_mio,prev_season_mv,two_seasons_ago_mv,mv_history_count,market_value_category
0,3332,Wayne Rooney,/wayne-rooney/profil/spieler/3332,10,Attack,Unknown,32.0,24/10/1985,England,1.76,...,Premier League,1,2017,€10.00m,10.0,381.25,NaN,NaN,0,Rendah
1,3333,James Milner,/james-milner/profil/spieler/3333,7,Midfield,Unknown,32.0,04/01/1986,England,1.75,...,Premier League,1,2017,€15.00m,15.0,857.50,NaN,NaN,0,Menengah
2,3333,James Milner,/james-milner/profil/spieler/3333,7,Midfield,Unknown,33.0,04/01/1986,England,1.75,...,Premier League,1,2018,€15.00m,15.0,1170.00,15.0,NaN,1,Menengah
3,3333,James Milner,/james-milner/profil/spieler/3333,7,Midfield,Unknown,34.0,04/01/1986,England,1.75,...,Premier League,1,2019,€6.50m,6.5,1000.00,15.0,15.0,2,Rendah
4,4360,Arjen Robben,/arjen-robben/profil/spieler/4360,10,Attack,Unknown,34.0,23/01/1984,Netherlands,1.80,...,Bundesliga,3,2017,€7.00m,7.0,811.85,NaN,NaN,0,Rendah


## 5. Feature Engineering Aman

In [6]:
def add_safe_features(df):
    featured = df.copy()

    featured['has_prev_mv'] = featured['prev_season_mv'].notna().astype(int)
    featured['prev_growth_rate'] = safe_divide(
        featured['prev_season_mv'] - featured['two_seasons_ago_mv'],
        featured['two_seasons_ago_mv']
    )
    featured['prev_growth_rate_clipped'] = featured['prev_growth_rate'].clip(-5, 5)
    featured['prev_season_mv_log'] = np.log1p(featured['prev_season_mv'].fillna(0).clip(lower=0))
    featured['two_seasons_ago_mv_log'] = np.log1p(featured['two_seasons_ago_mv'].fillna(0).clip(lower=0))
    featured['prev_mv_category'] = featured['prev_season_mv'].apply(assign_previous_value_category)
    featured['two_seasons_ago_mv_category'] = featured['two_seasons_ago_mv'].apply(assign_previous_value_category)
    featured['prev_mv_distance_to_10'] = featured['prev_season_mv'].fillna(0) - 10
    featured['prev_mv_distance_to_30'] = featured['prev_season_mv'].fillna(0) - 30
    featured['prev_mv_to_club_total_ratio'] = safe_divide(featured['prev_season_mv'].fillna(0), featured['club_total_mv_mio'])
    featured['age_prev_mv_interaction'] = featured['age'] * featured['prev_season_mv'].fillna(0)

    featured['age_squared'] = featured['age'] ** 2
    featured['age_group'] = pd.cut(
        featured['age'],
        bins=[0, 20, 23, 27, 30, 35, 100],
        labels=['u20', '21_23', '24_27', '28_30', '31_35', '36_plus'],
        include_lowest=True
    ).astype('object').fillna('Unknown')
    featured['age_peak_distance'] = (featured['age'] - 27).abs()
    featured['is_peak_age'] = featured['age'].between(24, 29).astype(int)

    position_lower = featured['pos_category'].astype(str).str.lower()
    featured['is_goalkeeper'] = position_lower.str.contains('goalkeeper|keeper').astype(int)
    featured['is_defender'] = position_lower.str.contains('defender|defence|defense').astype(int)
    featured['is_midfielder'] = position_lower.str.contains('midfield|midfielder').astype(int)
    featured['is_forward'] = position_lower.str.contains('attack|forward|striker|winger').astype(int)

    featured['club_total_mv_log'] = np.log1p(featured['club_total_mv_mio'].fillna(0).clip(lower=0))
    featured['club_total_mv_rank_league_season'] = featured.groupby(['league', 'season'])['club_total_mv_mio'].rank(
        method='dense', ascending=False
    )
    featured['club_total_mv_pct_league_season'] = featured.groupby(['league', 'season'])['club_total_mv_mio'].rank(
        method='average', ascending=False, pct=True
    )

    league_avg = featured.groupby(['league', 'season'])['club_total_mv_mio'].transform('mean')
    league_median = featured.groupby(['league', 'season'])['club_total_mv_mio'].transform('median')
    featured['club_mv_relative_to_league_avg'] = safe_divide(featured['club_total_mv_mio'], league_avg)
    featured['club_mv_relative_to_league_median'] = safe_divide(featured['club_total_mv_mio'], league_median)
    featured['age_position_interaction'] = featured['age_group'].astype(str) + '_' + featured['pos_category'].astype(str)
    featured['prev_mv_position_rank'] = featured.groupby(['pos_category', 'season'])['prev_season_mv'].rank(
        method='average', ascending=False
    ).fillna(0)
    featured['prev_mv_league_season_rank'] = featured.groupby(['league', 'season'])['prev_season_mv'].rank(
        method='average', ascending=False
    ).fillna(0)
    featured['prev_mv_club_ratio_category'] = pd.cut(
        featured['prev_mv_to_club_total_ratio'],
        bins=[-np.inf, 0, 0.02, 0.05, 0.10, np.inf],
        labels=['no_history', 'low', 'medium', 'high', 'very_high']
    ).astype('object').fillna('no_history')

    historical_fill_zero = [
        'prev_season_mv', 'two_seasons_ago_mv', 'prev_growth_rate',
        'prev_growth_rate_clipped', 'prev_season_mv_log', 'two_seasons_ago_mv_log',
        'prev_mv_distance_to_10', 'prev_mv_distance_to_30',
        'prev_mv_to_club_total_ratio', 'age_prev_mv_interaction'
    ]
    for col in historical_fill_zero:
        featured[col] = featured[col].replace([np.inf, -np.inf], np.nan).fillna(0)

    return featured


transfermarkt_featured = add_safe_features(transfermarkt_clean)
transfermarkt_featured[[TARGET_COLUMN, 'prev_season_mv', 'has_prev_mv', 'club_total_mv_mio']].head()

,market_value_category,prev_season_mv,has_prev_mv,club_total_mv_mio
0,Rendah,0.0,0,381.25
1,Menengah,0.0,0,857.50
2,Menengah,15.0,1,1170.00
3,Rendah,15.0,1,1000.00
4,Rendah,0.0,0,811.85


## 6. Mapping dan Matching FBref

In [7]:
def first_available(df, columns):
    existing = [col for col in columns if col in df.columns]
    if not existing:
        return pd.Series(0, index=df.index, dtype='float64')
    result = pd.to_numeric(df[existing[0]], errors='coerce')
    for col in existing[1:]:
        result = result.fillna(pd.to_numeric(df[col], errors='coerce'))
    return result.fillna(0)


def prepare_fbref_features(fbref):
    fb = fbref.copy()
    required = ['season', 'team', 'player']
    validate_columns(fb, required, 'fbref_player_stats.csv')

    fb['season'] = pd.to_numeric(fb['season'], errors='coerce')
    fb = fb[fb['season'].between(MIN_SEASON, MAX_SEASON)].copy()
    fb['player_key'] = fb['player'].apply(normalize_text)
    fb['club_key'] = fb['team'].apply(normalize_text)

    perf = pd.DataFrame(index=fb.index)
    perf['matches_played'] = first_available(fb, ['standard_playing_time_mp', 'keeper_playing_time_mp'])
    perf['starts'] = first_available(fb, ['standard_playing_time_starts', 'keeper_playing_time_starts'])
    perf['minutes'] = first_available(fb, ['standard_playing_time_min', 'keeper_playing_time_min'])
    perf['goals'] = first_available(fb, ['standard_performance_gls', 'shooting_standard_gls'])
    perf['assists'] = first_available(fb, ['standard_performance_ast'])
    perf['non_penalty_goals'] = first_available(fb, ['standard_performance_g_pk'])
    perf['yellow_cards'] = first_available(fb, ['standard_performance_crdy', 'misc_performance_crdy'])
    perf['red_cards'] = first_available(fb, ['standard_performance_crdr', 'misc_performance_crdr'])
    perf['shots_total'] = first_available(fb, ['shooting_standard_sh'])
    perf['shots_on_target'] = first_available(fb, ['shooting_standard_sot'])
    perf['fouls_committed'] = first_available(fb, ['misc_performance_fls'])
    perf['fouls_drawn'] = first_available(fb, ['misc_performance_fld'])
    perf['saves'] = first_available(fb, ['keeper_performance_saves'])
    perf['clean_sheets'] = first_available(fb, ['keeper_performance_cs'])
    perf['goals_against'] = first_available(fb, ['keeper_performance_ga'])
    perf['shots_on_target_against'] = first_available(fb, ['keeper_performance_sota'])

    nineties = perf['minutes'] / 90
    perf['goals_per_90'] = safe_divide(perf['goals'], nineties)
    perf['assists_per_90'] = safe_divide(perf['assists'], nineties)
    perf['goal_assist_per_90'] = safe_divide(perf['goals'] + perf['assists'], nineties)
    perf['shots_per_90'] = safe_divide(perf['shots_total'], nineties)
    perf['shots_on_target_per_90'] = safe_divide(perf['shots_on_target'], nineties)
    perf['cards_per_90'] = safe_divide(perf['yellow_cards'] + perf['red_cards'], nineties)
    perf['starts_rate'] = safe_divide(perf['starts'], perf['matches_played'])
    perf['save_pct'] = safe_divide(perf['saves'], perf['shots_on_target_against'])
    perf['clean_sheet_pct'] = safe_divide(perf['clean_sheets'], perf['matches_played'])
    perf['has_performance_stats'] = 1
    perf['goal_contribution_per_90'] = perf['goal_assist_per_90']
    perf['attacking_output_index'] = perf['goal_assist_per_90'] + perf['shots_on_target_per_90']
    perf['discipline_index'] = perf['cards_per_90'] + safe_divide(perf['fouls_committed'], nineties)
    perf['goalkeeper_output_index'] = perf['save_pct'] + perf['clean_sheet_pct']

    result = pd.concat([
        fb[['season', 'team', 'player', 'player_key', 'club_key']].reset_index(drop=True),
        perf[FBREF_FEATURES].reset_index(drop=True)
    ], axis=1)
    return result


def empty_matching_audit(tm_df, reason):
    audit = tm_df[['row_id', 'player_id', 'player_name', 'club', 'league', 'season']].copy()
    audit['matched'] = False
    audit['match_type'] = 'none'
    audit['name_score'] = 0
    audit['club_score'] = 0
    audit['fbref_player_name'] = ''
    audit['fbref_club'] = ''
    audit['reason'] = reason
    return audit


def merge_fbref_features(tm_df, fbref):
    merged = tm_df.copy().reset_index(drop=True)
    merged['row_id'] = np.arange(len(merged))
    merged['player_key'] = merged['player_name'].apply(normalize_text)
    merged['club_key'] = merged['club'].apply(normalize_text)

    for col in FBREF_FEATURES:
        merged[col] = 0

    if fbref.empty:
        audit = empty_matching_audit(merged, 'fbref_unavailable')
        return merged.drop(columns=['player_key', 'club_key']), audit

    fb = prepare_fbref_features(fbref)
    if fb.empty:
        audit = empty_matching_audit(merged, 'fbref_empty_after_filter')
        return merged.drop(columns=['player_key', 'club_key']), audit

    exact_map = {}
    season_keys = {}
    for season, frame in fb.groupby('season'):
        by_name = {}
        for record in frame.to_dict('records'):
            by_name.setdefault(record['player_key'], []).append(record)
        exact_map[season] = by_name
        season_keys[season] = sorted(by_name.keys())

    fuzzy_name_cache = {}
    audit_rows = []
    matched_feature_rows = {}

    def choose_best_by_club(records, tm_club_key):
        best_record = None
        best_club_score = -1
        for record in records:
            club_score_value = fuzz.token_sort_ratio(tm_club_key, record['club_key'])
            if club_score_value > best_club_score:
                best_record = record
                best_club_score = club_score_value
        return best_record, int(best_club_score)

    for row in merged.to_dict('records'):
        season = row['season']
        by_name = exact_map.get(season, {})
        match = None
        match_type = 'none'
        name_score = 0
        club_score = 0
        reason = 'no_candidate_same_season'

        exact_records = by_name.get(row['player_key'], [])
        if exact_records:
            match, club_score = choose_best_by_club(exact_records, row['club_key'])
            match_type = 'exact_player_key_season'
            name_score = 100
        elif by_name:
            cache_key = (season, row['player_key'])
            if cache_key not in fuzzy_name_cache:
                fuzzy_name_cache[cache_key] = process.extract(
                    row['player_key'], season_keys.get(season, []), scorer=fuzz.WRatio, score_cutoff=88, limit=5
                )
            fuzzy_candidates = fuzzy_name_cache[cache_key]
            best_record = None
            best_name_score = 0
            best_club_score = 0
            best_combined_score = -1
            for candidate_key, candidate_name_score, _ in fuzzy_candidates:
                candidate_records = by_name.get(candidate_key, [])
                local_record, local_club_score = choose_best_by_club(candidate_records, row['club_key'])
                combined_score = candidate_name_score + local_club_score
                if local_record is not None and combined_score > best_combined_score:
                    best_record = local_record
                    best_name_score = int(candidate_name_score)
                    best_club_score = int(local_club_score)
                    best_combined_score = combined_score

            if best_record is not None:
                name_score = best_name_score
                club_score = best_club_score
                if name_score >= 88 and club_score >= 70:
                    match = best_record
                    match_type = 'fuzzy_player_key_season_club'
                elif name_score >= 95:
                    match = best_record
                    match_type = 'fuzzy_player_key_strict_season'
                else:
                    reason = 'score_below_threshold'
            else:
                reason = 'no_fuzzy_candidate'

        if match is not None:
            matched_feature_rows[int(row['row_id'])] = {col: match[col] for col in FBREF_FEATURES}
            audit_rows.append({
                'row_id': row['row_id'], 'player_id': row['player_id'],
                'player_name': row['player_name'], 'club': row['club'],
                'league': row['league'], 'season': row['season'],
                'matched': True, 'match_type': match_type,
                'name_score': name_score, 'club_score': club_score,
                'fbref_player_name': match['player'], 'fbref_club': match['team'],
                'reason': 'matched'
            })
        else:
            audit_rows.append({
                'row_id': row['row_id'], 'player_id': row['player_id'],
                'player_name': row['player_name'], 'club': row['club'],
                'league': row['league'], 'season': row['season'],
                'matched': False, 'match_type': 'none',
                'name_score': 0, 'club_score': 0,
                'fbref_player_name': '', 'fbref_club': '', 'reason': reason
            })

    feature_frame = pd.DataFrame.from_dict(matched_feature_rows, orient='index')
    if not feature_frame.empty:
        feature_frame.index.name = 'row_id'
        feature_frame = feature_frame.reset_index()
        merged = merged.drop(columns=FBREF_FEATURES).merge(feature_frame, on='row_id', how='left')

    audit = pd.DataFrame(audit_rows)
    merged['has_performance_stats'] = merged['row_id'].isin(matched_feature_rows.keys()).astype(int)
    for col in FBREF_FEATURES:
        if col not in merged.columns:
            merged[col] = 0
        merged[col] = pd.to_numeric(merged[col], errors='coerce').fillna(0)

    return merged.drop(columns=['player_key', 'club_key']), audit


dataset_with_fbref, matching_audit = merge_fbref_features(transfermarkt_featured, fbref_raw)
print('Rows matched FBref:', int(matching_audit['matched'].sum()))
print('Rows unmatched FBref:', int((~matching_audit['matched']).sum()))

Rows matched FBref: 9836
Rows unmatched FBref: 375


## 7. Build Dataset Model

In [8]:
PROFILE_FEATURES = [
    'age', 'age_peak_distance', 'is_peak_age', 'height_m',
    'preferred_foot', 'pos_category', 'is_goalkeeper', 'is_defender',
    'is_midfielder', 'is_forward', 'league', 'league_rank'
]

CLUB_FEATURES = [
    'club_total_mv_mio', 'club_total_mv_rank_league_season',
    'club_total_mv_pct_league_season', 'club_mv_relative_to_league_avg'
]

HISTORY_FEATURES = [
    'prev_season_mv', 'two_seasons_ago_mv', 'has_prev_mv',
    'mv_history_count', 'prev_growth_rate_clipped', 'prev_season_mv_log',
    'prev_mv_category', 'two_seasons_ago_mv_category',
    'prev_mv_to_club_total_ratio'
]

PERFORMANCE_FEATURES = [
    'minutes', 'goals', 'assists', 'yellow_cards', 'shots_total',
    'shots_on_target', 'starts_rate', 'goals_per_90', 'assists_per_90',
    'goal_assist_per_90', 'shots_per_90', 'has_performance_stats'
]

MODEL_FEATURE_COLUMNS = PROFILE_FEATURES + CLUB_FEATURES + HISTORY_FEATURES + PERFORMANCE_FEATURES

missing_model_columns = [col for col in MODEL_FEATURE_COLUMNS if col not in dataset_with_fbref.columns]
if missing_model_columns:
    raise ValueError(f'Kolom fitur model belum tersedia: {missing_model_columns}')

feature_list = MODEL_FEATURE_COLUMNS.copy()
metadata_columns = ['season']
model_columns = feature_list + [col for col in metadata_columns if col not in feature_list] + [TARGET_COLUMN]
model_dataset = dataset_with_fbref[model_columns].copy()

for col in model_dataset.columns:
    if col == TARGET_COLUMN:
        continue
    if model_dataset[col].dtype == 'object' or str(model_dataset[col].dtype).startswith('category'):
        model_dataset[col] = model_dataset[col].astype('object').where(model_dataset[col].notna(), 'Unknown')
    else:
        model_dataset[col] = pd.to_numeric(model_dataset[col], errors='coerce').replace([np.inf, -np.inf], np.nan).fillna(0)

dataset_summary = pd.DataFrame([
    {'metric': 'model_rows', 'value': len(model_dataset)},
    {'metric': 'model_features', 'value': len(feature_list)},
    {'metric': 'clean_rows', 'value': len(dataset_with_fbref)},
    {'metric': 'fbref_matched_rows', 'value': int(matching_audit['matched'].sum())},
    {'metric': 'fbref_unmatched_rows', 'value': int((~matching_audit['matched']).sum())},
    {'metric': 'train_rows_planned', 'value': int(model_dataset['season'].between(2017, 2021).sum())},
    {'metric': 'validation_rows_planned', 'value': int((model_dataset['season'] == 2022).sum())},
    {'metric': 'test_rows_planned', 'value': int(model_dataset['season'].between(2023, 2024).sum())},
])

model_dataset.head()

,age,age_peak_distance,is_peak_age,height_m,preferred_foot,pos_category,is_goalkeeper,is_defender,is_midfielder,is_forward,...,shots_total,shots_on_target,starts_rate,goals_per_90,assists_per_90,goal_assist_per_90,shots_per_90,has_performance_stats,season,market_value_category
0,32.0,5.0,0,1.76,right,Attack,0,0,0,1,...,45.0,18.0,0.870968,0.396825,0.079365,0.476190,1.785714,1,2017,Rendah
1,32.0,5.0,0,1.75,right,Midfield,0,0,1,0,...,21.0,5.0,0.500000,0.000000,0.152113,0.152113,1.064789,1,2017,Menengah
2,33.0,6.0,0,1.75,right,Midfield,0,0,1,0,...,24.0,8.0,0.612903,0.251537,0.201230,0.452767,1.207378,1,2018,Menengah
3,34.0,7.0,0,1.75,right,Midfield,0,0,1,0,...,12.0,4.0,0.409091,0.192102,0.192102,0.384205,1.152615,1,2019,Rendah
4,34.0,7.0,0,1.80,left,Attack,0,0,0,1,...,51.0,21.0,0.904762,0.298607,0.298607,0.597213,3.045786,1,2017,Rendah


## 8. Validasi Anti Target Leakage

In [9]:
def validate_preprocessing_outputs(clean_df, model_df, audit_df, feature_columns):
    if clean_df.empty:
        raise ValueError('Clean dataset kosong.')
    if model_df.empty:
        raise ValueError('Model dataset kosong.')
    if not clean_df['season'].between(MIN_SEASON, MAX_SEASON).all():
        raise ValueError('Clean dataset memiliki season di luar 2017 sampai 2024.')
    if not (clean_df['market_value_mio'] >= 5).all():
        raise ValueError('Clean dataset masih memiliki market_value_mio di bawah 5.')
    invalid_targets = sorted(set(clean_df[TARGET_COLUMN].dropna()) - set(TARGET_LABELS))
    if invalid_targets:
        raise ValueError(f'Target tidak valid pada clean dataset: {invalid_targets}')
    if clean_df.duplicated(['player_id', 'season']).any():
        raise ValueError('Duplikasi player_id + season masih ditemukan pada clean dataset.')
    if TARGET_COLUMN not in model_df.columns:
        raise ValueError('Dataset model tidak memiliki target column.')

    feature_set = set(feature_columns)
    forbidden_in_features = sorted(feature_set.intersection(FORBIDDEN_FEATURES))
    if forbidden_in_features:
        raise ValueError(f'Forbidden feature masuk ke fitur model: {forbidden_in_features}')
    if 'mv_growth_rate' in model_df.columns:
        raise ValueError('Fitur terlarang mv_growth_rate ditemukan.')

    train_rows = model_df['season'].between(2017, 2021).sum()
    validation_rows = (model_df['season'] == 2022).sum()
    test_rows = model_df['season'].between(2023, 2024).sum()
    if train_rows == 0 or validation_rows == 0 or test_rows == 0:
        raise ValueError(
            f'Split kosong. train={train_rows}, validation={validation_rows}, test={test_rows}'
        )

    missing_fbref_features = [col for col in PERFORMANCE_FEATURES if col not in model_df.columns]
    if missing_fbref_features:
        raise ValueError(f'Fitur performa tidak tersedia: {missing_fbref_features}')
    if audit_df.empty:
        raise ValueError('Audit matching FBref kosong.')

    return {
        'train_rows': int(train_rows),
        'validation_rows': int(validation_rows),
        'test_rows': int(test_rows),
        'feature_count': len(feature_columns),
        'fbref_matched_rows': int(audit_df['matched'].sum()),
        'fbref_unmatched_rows': int((~audit_df['matched']).sum()),
    }


validation_summary = validate_preprocessing_outputs(
    dataset_with_fbref, model_dataset, matching_audit, feature_list
)
validation_summary

{'train_rows': 6138,
 'validation_rows': 1334,
 'test_rows': 2739,
 'feature_count': 37,
 'fbref_matched_rows': 9836,
 'fbref_unmatched_rows': 375}

## 9. Simpan Output Final

In [10]:
unmatched_players = matching_audit[~matching_audit['matched']].copy()

preprocessing_report.extend([
    {'metric': 'fbref_available', 'value': bool(fbref_available)},
    {'metric': 'fbref_matched_rows', 'value': validation_summary['fbref_matched_rows']},
    {'metric': 'fbref_unmatched_rows', 'value': validation_summary['fbref_unmatched_rows']},
    {'metric': 'fbref_fallback_used', 'value': not bool(fbref_available)},
    {'metric': 'model_rows', 'value': len(model_dataset)},
    {'metric': 'model_features', 'value': len(feature_list)},
    {'metric': 'train_rows_planned', 'value': validation_summary['train_rows']},
    {'metric': 'validation_rows_planned', 'value': validation_summary['validation_rows']},
    {'metric': 'test_rows_planned', 'value': validation_summary['test_rows']},
])

report_df = pd.DataFrame(preprocessing_report)

dataset_with_fbref.to_csv(CLEAN_FILE, index=False)
report_df.to_csv(PREPROCESSING_REPORT_FILE, index=False)
model_dataset.to_csv(MODEL_FILE, index=False)
dataset_summary.to_csv(DATASET_SUMMARY_FILE, index=False)
matching_audit.to_csv(MATCHING_FILE, index=False)
unmatched_players.to_csv(UNMATCHED_FILE, index=False)

with open(FEATURE_LIST_FILE, 'w', encoding='utf-8') as file:
    json.dump(feature_list, file, indent=2)

required_outputs = [
    CLEAN_FILE, PREPROCESSING_REPORT_FILE, MODEL_FILE, FEATURE_LIST_FILE,
    DATASET_SUMMARY_FILE, MATCHING_FILE, UNMATCHED_FILE
]
missing_outputs = [str(path) for path in required_outputs if not path.exists() or path.stat().st_size == 0]
if missing_outputs:
    raise ValueError(f'Output preprocessing gagal dibuat: {missing_outputs}')

print('Preprocessing selesai.')
print('Clean dataset        :', CLEAN_FILE)
print('Model dataset        :', MODEL_FILE)
print('Feature list         :', FEATURE_LIST_FILE)
print('Matching audit       :', MATCHING_FILE)
print('Rows model           :', len(model_dataset))
print('Jumlah fitur         :', len(feature_list))
print('Train rows planned   :', validation_summary['train_rows'])
print('Validation rows      :', validation_summary['validation_rows'])
print('Test rows planned    :', validation_summary['test_rows'])
print('FBref matched rows   :', validation_summary['fbref_matched_rows'])
print('FBref unmatched rows :', validation_summary['fbref_unmatched_rows'])

Preprocessing selesai.
Clean dataset        : c:\KULIAH\SEMESTER 6\BIG DATA\UAS\Tugas_UAS_Big-Data\data\processed\transfermarkt_clean.csv
Model dataset        : c:\KULIAH\SEMESTER 6\BIG DATA\UAS\Tugas_UAS_Big-Data\data\model\players_model.csv
Feature list         : c:\KULIAH\SEMESTER 6\BIG DATA\UAS\Tugas_UAS_Big-Data\data\model\feature_list.json
Matching audit       : c:\KULIAH\SEMESTER 6\BIG DATA\UAS\Tugas_UAS_Big-Data\data\interim\player_matching_result.csv
Rows model           : 10211
Jumlah fitur         : 37
Train rows planned   : 6138
Validation rows      : 1334
Test rows planned    : 2739
FBref matched rows   : 9836
FBref unmatched rows : 375


## 10. Ringkasan Dataset

In [11]:
display(report_df)
display(model_dataset[TARGET_COLUMN].value_counts().rename_axis('label').reset_index(name='rows'))
display(model_dataset.groupby('season').size().rename('rows').reset_index())

,metric,value
0,raw_rows,30024
1,rows_after_season_filter,30024
2,rows_dropped_missing_market_value,1690
3,duplicate_player_season_rows_dropped,1593
4,rows_dropped_below_5_mio,16530
5,clean_rows,10211
6,fbref_available,True
7,fbref_matched_rows,9836
8,fbref_unmatched_rows,375
9,fbref_fallback_used,False


,label,rows
0,Rendah,5534
1,Menengah,3395
2,Tinggi,1282


,season,rows
0,2017,1119
1,2018,1306
2,2019,1172
3,2020,1284
4,2021,1257
5,2022,1334
6,2023,1342
7,2024,1397
